# MQTT Publisher 測試應用程式

這個 notebook 用於測試 MQTT 訊息發布功能。


In [ ]:
# ⚠️ 重要：請先執行這個 cell 安裝套件！
# 如果遇到 ModuleNotFoundError，請先執行此 cell 再執行其他 cell

%pip install paho-mqtt

# 驗證安裝
try:
    import paho.mqtt.client as mqtt
    print("✅ paho-mqtt 安裝成功！")
except ImportError as e:
    print(f"❌ 安裝失敗: {e}")
    print("請重試安裝或檢查網路連線")


## ⚠️ 重要提醒

**執行下面的程式碼之前，請務必先執行上方的安裝 cell (Cell 1)**，否則會出現 `ModuleNotFoundError` 錯誤。

如果遇到錯誤，請：
1. 先執行 Cell 1 安裝套件
2. 等待安裝完成
3. 再執行下面的範例程式碼


## 基本 MQTT Publisher

簡單的發布單一訊息範例：


In [ ]:
# 檢查並導入必要的套件
try:
    import paho.mqtt.client as mqtt
    import time
    from datetime import datetime
    print("✅ 套件導入成功！")
except ImportError as e:
    print(f"❌ 導入失敗: {e}")
    print("\n💡 請先執行 Cell 1 安裝 paho-mqtt 套件！")
    print("   執行: %pip install paho-mqtt")
    raise  # 停止執行，避免後續錯誤

# ============ MQTT 設定 ============
BROKER = "localhost"      # MQTT Broker 位址 (例如: "192.168.1.100" 或 "mqtt.eclipse.org")
PORT = 1883               # MQTT 埠號
TOPIC = "test/raspberry"  # 發布的主題
CLIENT_ID = "raspberry_publisher"  # 客戶端 ID

# ============ 建立 MQTT 客戶端 ============
client = mqtt.Client(client_id=CLIENT_ID)

# 連線成功回呼函數
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print(f"✅ 成功連線到 MQTT Broker: {BROKER}:{PORT}")
    else:
        print(f"❌ 連線失敗，錯誤代碼: {rc}")

# 發布成功回呼函數
def on_publish(client, userdata, mid):
    print(f"📤 訊息發布成功 (訊息 ID: {mid})")

# 設定回呼函數
client.on_connect = on_connect
client.on_publish = on_publish

# ============ 連線並發布訊息 ============
try:
    print(f"🔌 嘗試連線到 {BROKER}:{PORT}...")
    client.connect(BROKER, PORT, 60)
    client.loop_start()  # 開始網路循環（非阻塞）
    
    # 等待連線建立
    time.sleep(1)
    
    # 發布測試訊息
    message = f"Hello MQTT! 時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    result = client.publish(TOPIC, message, qos=1)
    
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"📝 準備發布訊息到主題: {TOPIC}")
        print(f"📨 訊息內容: {message}")
        result.wait_for_publish()  # 等待發布完成
    else:
        print(f"❌ 發布失敗，錯誤代碼: {result.rc}")
    
    # 等待訊息發送完成
    time.sleep(2)
    
    # 斷線
    client.loop_stop()
    client.disconnect()
    print("🔌 已斷線")
    
except Exception as e:
    print(f"❌ 發生錯誤: {e}")


ModuleNotFoundError: No module named 'paho'

## 循環發布訊息（測試用）

每隔幾秒自動發布一次訊息，可用於持續測試：


In [ ]:
import paho.mqtt.client as mqtt
import time
from datetime import datetime

# ============ MQTT 設定 ============
BROKER = "localhost"
PORT = 1883
TOPIC = "test/raspberry"
CLIENT_ID = "raspberry_publisher_loop"

# ============ 發布設定 ============
PUBLISH_INTERVAL = 3  # 每隔幾秒發布一次（秒）
TOTAL_MESSAGES = 5    # 總共發布幾次訊息（設為 0 表示無限循環，按 Ctrl+C 停止）

# ============ 建立 MQTT 客戶端 ============
client = mqtt.Client(client_id=CLIENT_ID)

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print(f"✅ 成功連線到 MQTT Broker: {BROKER}:{PORT}")
    else:
        print(f"❌ 連線失敗，錯誤代碼: {rc}")

def on_publish(client, userdata, mid):
    print(f"  ✓ 發布成功 (ID: {mid})")

client.on_connect = on_connect
client.on_publish = on_publish

# ============ 開始循環發布 ============
try:
    print(f"🔌 連線到 {BROKER}:{PORT}...")
    client.connect(BROKER, PORT, 60)
    client.loop_start()
    time.sleep(1)
    
    count = 0
    print(f"\n🚀 開始發布訊息 (每隔 {PUBLISH_INTERVAL} 秒)...")
    print("=" * 50)
    
    while True:
        count += 1
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        message = f"測試訊息 #{count} - {timestamp}"
        
        print(f"\n[{count}] 📤 發布到主題: {TOPIC}")
        print(f"     📨 內容: {message}")
        
        result = client.publish(TOPIC, message, qos=1)
        
        if TOTAL_MESSAGES > 0 and count >= TOTAL_MESSAGES:
            print(f"\n✅ 已完成 {TOTAL_MESSAGES} 次發布")
            break
        
        time.sleep(PUBLISH_INTERVAL)
    
    time.sleep(2)
    client.loop_stop()
    client.disconnect()
    print("\n🔌 已斷線")
    
except KeyboardInterrupt:
    print("\n\n⚠️  使用者中斷")
    client.loop_stop()
    client.disconnect()
    print("🔌 已斷線")
    
except Exception as e:
    print(f"\n❌ 發生錯誤: {e}")
    client.loop_stop()
    client.disconnect()


## 進階版本：可自訂訊息的 Publisher

更靈活的版本，可以自訂各種參數：


In [ ]:
import paho.mqtt.client as mqtt
import time
import json
from datetime import datetime

class MQTTPublisher:
    """MQTT Publisher 類別，方便測試使用"""
    
    def __init__(self, broker="localhost", port=1883, client_id=None):
        self.broker = broker
        self.port = port
        self.client_id = client_id or f"publisher_{int(time.time())}"
        self.client = mqtt.Client(client_id=self.client_id)
        self.client.on_connect = self._on_connect
        self.client.on_publish = self._on_publish
        self.connected = False
    
    def _on_connect(self, client, userdata, flags, rc):
        if rc == 0:
            self.connected = True
            print(f"✅ 連線成功: {self.broker}:{self.port}")
        else:
            print(f"❌ 連線失敗，錯誤代碼: {rc}")
    
    def _on_publish(self, client, userdata, mid):
        print(f"  ✓ 發布成功 (ID: {mid})")
    
    def connect(self):
        """連線到 MQTT Broker"""
        try:
            self.client.connect(self.broker, self.port, 60)
            self.client.loop_start()
            time.sleep(1)  # 等待連線
            return self.connected
        except Exception as e:
            print(f"❌ 連線錯誤: {e}")
            return False
    
    def publish(self, topic, message, qos=1, retain=False):
        """發布訊息"""
        if not self.connected:
            print("⚠️  尚未連線，請先呼叫 connect()")
            return False
        
        result = self.client.publish(topic, message, qos=qos, retain=retain)
        return result.rc == mqtt.MQTT_ERR_SUCCESS
    
    def publish_json(self, topic, data, qos=1, retain=False):
        """發布 JSON 格式訊息"""
        json_message = json.dumps(data, ensure_ascii=False)
        return self.publish(topic, json_message, qos, retain)
    
    def disconnect(self):
        """斷線"""
        if self.connected:
            self.client.loop_stop()
            self.client.disconnect()
            print("🔌 已斷線")

# ============ 使用範例 ============
# 建立 Publisher
publisher = MQTTPublisher(
    broker="localhost",  # 修改為您的 MQTT Broker 位址
    port=1883
)

# 連線
if publisher.connect():
    # 發布簡單文字訊息
    publisher.publish("test/raspberry/simple", "Hello from Raspberry Pi!")
    time.sleep(1)
    
    # 發布 JSON 訊息
    data = {
        "device": "Raspberry Pi",
        "temperature": 25.5,
        "humidity": 60,
        "timestamp": datetime.now().isoformat()
    }
    publisher.publish_json("test/raspberry/json", data)
    time.sleep(1)
    
    # 發布帶有時間戳的訊息
    message = f"測試訊息 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    publisher.publish("test/raspberry/timestamp", message, qos=1)
    time.sleep(1)
    
    # 斷線
    publisher.disconnect()
else:
    print("無法連線到 MQTT Broker")


## 環境測試

檢查 MQTT 環境是否可以使用：


In [ ]:
import subprocess
import sys
import socket

print("=" * 60)
print("🔍 MQTT 環境檢查")
print("=" * 60)

# 1. 檢查 Mosquitto 是否安裝
print("\n1️⃣  檢查 Mosquitto Broker 安裝狀態...")
try:
    result = subprocess.run(['which', 'mosquitto'], 
                          capture_output=True, text=True, timeout=2)
    if result.returncode == 0:
        print(f"   ✅ Mosquitto 已安裝: {result.stdout.strip()}")
    else:
        print("   ❌ Mosquitto 未安裝")
except Exception as e:
    print(f"   ⚠️  檢查時發生錯誤: {e}")

# 2. 檢查 Mosquitto 服務狀態
print("\n2️⃣  檢查 Mosquitto 服務狀態...")
try:
    result = subprocess.run(['systemctl', 'is-active', 'mosquitto'], 
                          capture_output=True, text=True, timeout=2)
    if result.returncode == 0 and result.stdout.strip() == 'active':
        print("   ✅ Mosquitto 服務正在運行")
    else:
        print("   ⚠️  Mosquitto 服務未運行")
except Exception as e:
    print(f"   ⚠️  無法檢查服務狀態: {e}")

# 3. 檢查 MQTT 埠號是否開放
print("\n3️⃣  檢查 MQTT 埠號 (1883) 是否可用...")
try:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    result = sock.connect_ex(('localhost', 1883))
    sock.close()
    if result == 0:
        print("   ✅ MQTT 埠號 1883 可以連接")
    else:
        print("   ❌ MQTT 埠號 1883 無法連接")
except Exception as e:
    print(f"   ❌ 連接測試失敗: {e}")

# 4. 檢查 paho-mqtt 套件
print("\n4️⃣  檢查 paho-mqtt Python 套件...")
try:
    import paho.mqtt.client as mqtt
    print("   ✅ paho-mqtt 套件已安裝")
    try:
        print(f"   📦 版本資訊可用")
    except:
        pass
except ImportError:
    print("   ❌ paho-mqtt 套件未安裝")
    print("   💡 請執行: %pip install paho-mqtt")

# 5. 測試實際連線
print("\n5️⃣  測試 MQTT 連線...")
try:
    import paho.mqtt.client as mqtt
    client = mqtt.Client("test_client")
    connected = False
    
    def on_connect_test(client, userdata, flags, rc):
        global connected
        if rc == 0:
            connected = True
    
    client.on_connect = on_connect_test
    client.connect("localhost", 1883, 5)
    client.loop_start()
    
    import time
    time.sleep(1)
    
    if connected:
        print("   ✅ 成功連接到 MQTT Broker!")
        client.disconnect()
        client.loop_stop()
    else:
        print("   ❌ 無法連接到 MQTT Broker")
        client.loop_stop()
except ImportError:
    print("   ⏭️  跳過（需要先安裝 paho-mqtt）")
except Exception as e:
    print(f"   ❌ 連線測試失敗: {e}")

print("\n" + "=" * 60)
print("✅ 環境檢查完成！")
print("=" * 60)


In [ ]:
import paho.mqtt.client as mqtt
import time
import json
from datetime import datetime

class MQTTPublisher:
    """MQTT Publisher 類別，方便測試使用"""
    
    def __init__(self, broker="localhost", port=1883, client_id=None):
        self.broker = broker
        self.port = port
        self.client_id = client_id or f"publisher_{int(time.time())}"
        self.client = mqtt.Client(client_id=self.client_id)
        self.client.on_connect = self._on_connect
        self.client.on_publish = self._on_publish
        self.connected = False
    
    def _on_connect(self, client, userdata, flags, rc):
        if rc == 0:
            self.connected = True
            print(f"✅ 連線成功: {self.broker}:{self.port}")
        else:
            print(f"❌ 連線失敗，錯誤代碼: {rc}")
    
    def _on_publish(self, client, userdata, mid):
        print(f"  ✓ 發布成功 (ID: {mid})")
    
    def connect(self):
        """連線到 MQTT Broker"""
        try:
            self.client.connect(self.broker, self.port, 60)
            self.client.loop_start()
            time.sleep(1)  # 等待連線
            return self.connected
        except Exception as e:
            print(f"❌ 連線錯誤: {e}")
            return False
    
    def publish(self, topic, message, qos=1, retain=False):
        """發布訊息"""
        if not self.connected:
            print("⚠️  尚未連線，請先呼叫 connect()")
            return False
        
        result = self.client.publish(topic, message, qos=qos, retain=retain)
        return result.rc == mqtt.MQTT_ERR_SUCCESS
    
    def publish_json(self, topic, data, qos=1, retain=False):
        """發布 JSON 格式訊息"""
        json_message = json.dumps(data, ensure_ascii=False)
        return self.publish(topic, json_message, qos, retain)
    
    def disconnect(self):
        """斷線"""
        if self.connected:
            self.client.loop_stop()
            self.client.disconnect()
            print("🔌 已斷線")

# ============ 使用範例 ============
# 建立 Publisher
publisher = MQTTPublisher(
    broker="localhost",  # 修改為您的 MQTT Broker 位址
    port=1883
)

# 連線
if publisher.connect():
    # 發布簡單文字訊息
    publisher.publish("test/raspberry/simple", "Hello from Raspberry Pi!")
    time.sleep(1)
    
    # 發布 JSON 訊息
    data = {
        "device": "Raspberry Pi",
        "temperature": 25.5,
        "humidity": 60,
        "timestamp": datetime.now().isoformat()
    }
    publisher.publish_json("test/raspberry/json", data)
    time.sleep(1)
    
    # 發布帶有時間戳的訊息
    message = f"測試訊息 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    publisher.publish("test/raspberry/timestamp", message, qos=1)
    time.sleep(1)
    
    # 斷線
    publisher.disconnect()
else:
    print("無法連線到 MQTT Broker")
